In [3]:
!pip -q install sentence-transformers joblib

import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
from google.colab import drive
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Conectar Google Drive
drive.mount('/content/drive')

# Definir las rutas donde tu compañero guardó las cosas
RUTA_APF1 = Path("/content/drive/MyDrive/Sentilytics/APF1")
RUTA_APF2 = Path("/content/drive/MyDrive/Sentilytics/APF2")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Cargar el test que nunca han visto los modelos
test = pd.read_csv(RUTA_APF1 / "test_reservado.csv")
y_test = test["sentimiento"].to_numpy()

print("Examen final (Test) cargado.")
print("Cantidad de reseñas a evaluar:", len(test))

Examen final (Test) cargado.
Cantidad de reseñas a evaluar: 155


In [5]:
# --- COMPETIDOR 1: APF1 (TF-IDF) ---
vectorizador_tfidf = joblib.load(RUTA_APF1 / "vectorizador_tfidf.joblib")
modelo_tfidf = joblib.load(RUTA_APF1 / "regresion_logistica.joblib")

# Leer el umbral del APF1 (0.48 según el trabajo de tu compañero)
with open(RUTA_APF1 / "configuracion.json", encoding="utf-8") as f:
    umbral_tfidf = json.load(f)["umbral_negativo"]

# --- COMPETIDOR 2: APF2 (Sentence-BERT) ---
modelo_sbert = joblib.load(RUTA_APF2 / "regresion_logistica_sbert.joblib")
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

with open(RUTA_APF2 / "configuracion_apf2.json", encoding="utf-8") as f:
    umbral_sbert = json.load(f)["umbral_negativo"]

print("Modelos y umbrales cargados correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelos y umbrales cargados correctamente.


In [6]:
# Evaluando con APF1 (TF-IDF usa el texto limpio)
X_test_tfidf = vectorizador_tfidf.transform(test["texto_tfidf"])
idx_negativa = list(modelo_tfidf.classes_).index(0)
prob_test_neg_tfidf = modelo_tfidf.predict_proba(X_test_tfidf)[:, idx_negativa]

# Predicción: 0 si la probabilidad es mayor al umbral (0.48), si no 1
pred_test_tfidf = np.where(prob_test_neg_tfidf >= umbral_tfidf, 0, 1)

# Evaluando con APF2 (SBERT usa el texto original/contextual)
E_test = encoder.encode(test["texto_contextual"].astype(str).tolist(), show_progress_bar=True)
prob_test_neg_sbert = modelo_sbert.predict_proba(E_test)[:, idx_negativa]

# Predicción SBERT
pred_test_sbert = np.where(prob_test_neg_sbert >= umbral_sbert, 0, 1)

print("Predicciones completadas.")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Predicciones completadas.


In [7]:
def calcular_metricas(y_real, y_predicho):
    vn, fp, fn, vp = confusion_matrix(y_real, y_predicho, labels=[1, 0]).ravel()
    return {
        "F1 negativo": f1_score(y_real, y_predicho, pos_label=0, zero_division=0),
        "Recall negativo": recall_score(y_real, y_predicho, pos_label=0, zero_division=0),
        "Precisión negativa": precision_score(y_real, y_predicho, pos_label=0, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_real, y_predicho),
        "F1 Macro": f1_score(y_real, y_predicho, average="macro", zero_division=0),
        "Quejas detectadas (VP)": int(vp),
        "Quejas omitidas (FN)": int(fn)
    }

# Juntamos los resultados en una tabla
tabla_final = pd.DataFrame({
    "TF-IDF (APF1)": calcular_metricas(y_test, pred_test_tfidf),
    "Sentence-BERT (APF2)": calcular_metricas(y_test, pred_test_sbert)
}).T

print("TABLA DE EVALUACIÓN FINAL EN TEST:")
display(tabla_final.round(4))

TABLA DE EVALUACIÓN FINAL EN TEST:


,F1 negativo,Recall negativo,Precisión negativa,Balanced accuracy,F1 Macro,Quejas detectadas (VP),Quejas omitidas (FN)
TF-IDF (APF1),0.5484,0.6296,0.4857,0.7445,0.7177,17.0,10.0
Sentence-BERT (APF2),0.7869,0.8889,0.7059,0.9054,0.8673,24.0,3.0
